In [ ]:
# Team members: TODO enter the three names and student IDs truthfully.
# Execution: generated notebook; see saved outputs and verify Run All on the submission machine.
# Sources: https://arxiv.org/abs/1409.1556 ; https://www.kaggle.com/datasets/vesuvius13/formula-one-cars ; AI_USAGE.md


# F1 team image classification

This notebook includes the implementation layer by layer and can run without the `f1_cnn` package. It predicts one of the dataset teams for a photo with one principal car. It is not an object detector.

Generated code and commentary must be reviewed and explained in your own words. Write your PDF yourself as required by the assignment. Complete names, IDs, contributions and the AI prompt log truthfully.

Install `requirements.txt` in the notebook kernel environment before running. The default displays a completed local run when available. Set `RUN_TRAINING = True` to train a fresh full suite on the chosen dataset.

In [ ]:
from pathlib import Path
import json
import os
import torch
from IPython.display import display, Image as DisplayImage

PROJECT_ROOT = Path.cwd()  # Open the notebook from the project folder.
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.cache' / 'matplotlib'))
torch.set_num_threads(4)  # Bound CPU parallelism to keep desktop training responsive.
CLEANED_ROOT = PROJECT_ROOT / 'data' / 'cleaned'
DATA_ROOT = CLEANED_ROOT if CLEANED_ROOT.is_dir() and any(CLEANED_ROOT.iterdir()) else PROJECT_ROOT / 'data' / 'raw' / 'Formula One Cars'
RUN_TRAINING = False  # True runs baseline, augmentation, three development folds and one final test.
RUN_DIR = None  # Set to an existing run path, or leave None to find the latest completed run.
PHOTO_PATH = None  # Set to a new photo path for optional inference at the end.


## 1. Configuration

Immutable configuration records the hyperparameters with each checkpoint.

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path


def default_data_root():
    # Notebook builds define PROJECT_ROOT; package callers resolve from this file.
    project = globals().get('PROJECT_ROOT') or Path(__file__).resolve().parents[1]
    cleaned = project / 'data' / 'cleaned'
    if cleaned.is_dir() and any(cleaned.iterdir()):
        return cleaned
    return project / 'data' / 'raw' / 'Formula One Cars'


@dataclass(frozen=True)
class TrainingConfig:
    """Settings to record alongside every experiment."""

    image_size: int = 64
    batch_size: int = 32
    epochs: int = 20
    learning_rate: float = 0.001
    weight_decay: float = 0.0001
    dropout: float = 0.3
    seed: int = 42
    augment: bool = False
    patience: int = 6
    num_workers: int = 0
    cache_images: bool = True

    def __post_init__(self):
        if self.image_size < 16 or self.batch_size < 1 or self.epochs < 1:
            raise ValueError('image_size >= 16, batch_size >= 1 and epochs >= 1 are required.')
        if not 0 <= self.dropout < 1 or self.learning_rate <= 0 or self.weight_decay < 0:
            raise ValueError('Invalid dropout, learning rate or weight decay.')
        if self.patience < 1 or self.num_workers < 0:
            raise ValueError('patience >= 1 and num_workers >= 0 are required.')

    def to_dict(self):
        return asdict(self)


## 2. Data audit and preprocessing

The read-only audit detects identical decoded pixels, rejects conflicting labels and keeps class ordering. RGB conversion, EXIF orientation, resizing and fixed normalisation are identical at inference. Augmentations occur only in training. Exact hashing does not find related crops or frames: review these before final experiments.

In [ ]:
"""Audit images before making reproducible, stratified data splits."""

from collections import Counter
from dataclasses import dataclass
import hashlib
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps, UnidentifiedImageError
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split, StratifiedKFold


@dataclass(frozen=True)
class ImageRecord:
    path: Path
    label: int
    team: str
    digest: str


class DatasetAudit:
    """Discover classes; exclude all copies of images with conflicting labels."""

    def __init__(self, root):
        self.root = Path(root)
        self.class_names = []
        self.records = []
        self.invalid = []
        self.duplicates = []
        self.conflicts = []
        self.skipped = []
        self.warnings = []

    def scan(self):
        if not self.root.is_dir():
            raise FileNotFoundError(f"Extract data first and point DATA_ROOT at the team folders: {self.root}")
        self.records, self.invalid, self.duplicates = [], [], []
        self.conflicts = []
        self.skipped, self.warnings = [], []
        conflicting_digests = set()
        folders = sorted(p for p in self.root.iterdir() if p.is_dir() and not p.name.startswith('.'))
        if len(folders) < 2:
            raise ValueError("DATA_ROOT must directly contain at least two team folders (check ZIP nesting).")
        self.class_names = [p.name for p in folders]
        if {'train', 'test', 'val', 'validation'} & {n.lower() for n in self.class_names}:
            raise ValueError('Existing split folders detected. Set the root to team folders; do not mix an existing test set into training.')
        seen = {}
        for label, folder in enumerate(folders):
            for path in sorted(folder.rglob('*')):
                if not path.is_file() or path.name.startswith('.'):
                    continue
                try:
                    with Image.open(path) as source:
                        problems = []
                        if source.format != 'JPEG':
                            problems.append('non-JPEG')
                        if source.mode != 'RGB':
                            problems.append('non-RGB')
                        if path.suffix.lower() not in {'.jpg', '.jpeg'}:
                            problems.append('non-JPEG extension')
                        image = ImageOps.exif_transpose(source).convert('RGB')
                        # Hash decoded pixels so metadata-only copies are also detected.
                        digest = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()
                        if problems:
                            self.warnings.append({'path': str(path), 'issues': problems})
                except UnidentifiedImageError as error:
                    if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif', '.tif', '.tiff'}:
                        self.invalid.append((str(path), str(error)))
                    else:
                        self.skipped.append(str(path))
                    continue
                except (OSError, ValueError) as error:
                    self.invalid.append((str(path), str(error)))
                    continue
                if digest in seen:
                    previous = seen[digest]
                    if previous.label != label:
                        self.conflicts.append((str(previous.path), str(path)))
                        conflicting_digests.add(digest)
                        continue
                    self.duplicates.append((str(path), str(previous.path)))
                    continue
                record = ImageRecord(path, label, folder.name, digest)
                seen[digest] = record
                self.records.append(record)
        self.records = [r for r in self.records if r.digest not in conflicting_digests]
        counts = Counter(r.team for r in self.records)
        if any(counts[name] < 5 for name in self.class_names):
            raise ValueError(f"Need at least five unique readable images per team; found {dict(counts)}. Check folder layout.")
        return self

    def summary(self):
        return {'class_counts': dict(Counter(r.team for r in self.records)),
                'invalid_files': self.invalid, 'removed_duplicates': self.duplicates,
                'excluded_label_conflicts': self.conflicts, 'skipped_non_images': self.skipped,
                'format_warnings': self.warnings, 'photos_retained': len(self.records),
                'root': str(self.root.resolve())}


class F1ImageDataset(Dataset):
    def __init__(self, records, image_size=128, augment=False, cache_images=False):
        self.records = list(records)
        self.image_size = image_size
        self.cache_images = cache_images
        self.cache = {}
        steps = [transforms.Resize((image_size, image_size))]
        if augment:
            steps += [transforms.RandomHorizontalFlip(), transforms.RandomRotation(10)]
        # Fixed scaling avoids estimating preprocessing statistics from test images.
        steps += [transforms.ToTensor(), transforms.Normalize((0.5,) * 3, (0.5,) * 3)]
        self.transform = transforms.Compose(steps)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        if index in self.cache:
            image = self.cache[index].copy()
        else:
            with Image.open(record.path) as source:
                image = ImageOps.exif_transpose(source).convert('RGB')
            # Cache resized RGB pixels, never random augmentations, to reduce disk work.
            image = transforms.functional.resize(image, [self.image_size, self.image_size])
            if self.cache_images:
                self.cache[index] = image.copy()
        return self.transform(image), record.label


class F1DataModule:
    """Keep test images separate; augment only training images."""

    def __init__(self, records, config):
        self.records = list(records)
        self.config = config

    def split(self, test_fraction=0.15, validation_fraction=0.15):
        if not 0 < test_fraction < 1 or not 0 < validation_fraction < 1 - test_fraction:
            raise ValueError('Split fractions must be positive and sum to less than one.')
        train_val, test = train_test_split(
            self.records, test_size=test_fraction, random_state=self.config.seed,
            stratify=[r.label for r in self.records])
        train, val = train_test_split(
            train_val, test_size=validation_fraction / (1 - test_fraction),
            random_state=self.config.seed, stratify=[r.label for r in train_val])
        return {'train': train, 'validation': val, 'test': test}

    def folds(self, development_records, n_splits=3):
        """Pass only train + validation records here, never the held-out test set."""
        records = list(development_records)
        labels = [r.label for r in records]
        if min(Counter(labels).values()) < n_splits:
            raise ValueError('Each team needs at least n_splits development images.')
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=self.config.seed)
        for train, val in splitter.split(np.zeros(len(records)), labels):
            yield [records[i] for i in train], [records[i] for i in val]

    def loader(self, records, training=False):
        dataset = F1ImageDataset(records, self.config.image_size, training and self.config.augment,
                                 self.config.cache_images)
        return DataLoader(dataset, batch_size=self.config.batch_size, shuffle=training,
                          num_workers=self.config.num_workers,
                          generator=torch.Generator().manual_seed(self.config.seed))


## 3. Explicit CNN architecture

Three blocks each contain two explicitly constructed 3x3 convolutions and ReLU activations, followed by max pooling. Global average pooling, dropout and a linear layer return logits. This is a compact VGG-inspired network without batch normalisation or pretrained weights.

In [ ]:
import torch
from torch import nn


class F1TeamCNN(nn.Module):
    """Small VGG-inspired model, not a reproduction of VGG16 or pretrained model."""

    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        if num_classes < 2:
            raise ValueError('At least two teams are required.')
        # Each block learns spatial features, then halves spatial resolution.
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(128, num_classes))
        for layer in self.modules():
            if isinstance(layer, nn.Conv2d):
                nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
                nn.init.zeros_(layer.bias)
            elif isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, images):
        # Return raw logits: CrossEntropyLoss applies log-softmax internally.
        return self.classifier(self.features(images))


## 4. Loss optimiser and training

Cross-entropy accepts logits and integer class labels. Adam and weight decay update the parameters. Gradients are disabled for validation; the best validation-loss weights are restored after training or early stopping.

In [ ]:
import copy
import json
from pathlib import Path
import random
import time

import numpy as np
import torch
from torch import nn
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, balanced_accuracy_score


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class Trainer:
    def __init__(self, model, config, class_names, device=None):
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.model = model.to(self.device)
        self.config = config
        self.class_names = list(class_names)
        # One mutually exclusive team label per image: feed raw logits to cross entropy.
        self.loss_fn = nn.CrossEntropyLoss()
        # L2 weight decay penalises large weights; Adam adapts each parameter's step size.
        self.optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate,
                                          weight_decay=config.weight_decay)
        self.history = []
        self.best_epoch = None

    def _epoch(self, loader, training):
        self.model.train(training)
        total_loss, correct, count = 0.0, 0, 0
        with torch.set_grad_enabled(training):
            for images, targets in loader:
                images, targets = images.to(self.device), targets.to(self.device)
                if training:
                    # Gradients accumulate by default, so clear them for each mini-batch.
                    self.optimizer.zero_grad()
                logits = self.model(images)
                loss = self.loss_fn(logits, targets)
                if training:
                    # Backpropagate the loss, then apply the optimiser's parameter update.
                    loss.backward()
                    self.optimizer.step()
                # Weight batch means by batch size, including the smaller last batch.
                total_loss += loss.item() * targets.size(0)
                correct += (logits.argmax(1) == targets).sum().item()
                count += targets.size(0)
        if not count:
            raise ValueError('Cannot train or evaluate an empty loader.')
        return total_loss / count, correct / count

    def fit(self, train_loader, validation_loader, output_dir):
        if self.config.epochs < 1:
            raise ValueError('epochs must be at least one.')
        output = Path(output_dir)
        # Separate experiment directories prevent accidental replacement of results.
        output.mkdir(parents=True, exist_ok=False)
        best_loss, best_state = float('inf'), None
        stale_epochs = 0
        for epoch in range(1, self.config.epochs + 1):
            started = time.perf_counter()
            train_loss, train_acc = self._epoch(train_loader, True)
            val_loss, val_acc = self._epoch(validation_loader, False)
            row = dict(epoch=epoch, train_loss=train_loss, validation_loss=val_loss,
                       train_accuracy=train_acc, validation_accuracy=val_acc,
                       seconds=time.perf_counter() - started)
            self.history.append(row)
            print(row, flush=True)
            if val_loss < best_loss:
                stale_epochs = 0
                self.best_epoch = epoch
                best_loss = val_loss
                # Copy tensors so later optimiser steps cannot mutate the saved best weights.
                best_state = copy.deepcopy(self.model.state_dict())
                torch.save({'state_dict': best_state, 'class_names': self.class_names,
                            'config': self.config.to_dict(), 'epoch': epoch}, output / 'best.pt')
            else:
                stale_epochs += 1
            (output / 'history.json').write_text(json.dumps(self.history, indent=2))
            if stale_epochs >= self.config.patience:
                print(f'Early stopping; restoring epoch {self.best_epoch}.', flush=True)
                break
        if best_state is None:
            raise RuntimeError('No finite validation loss. Check images and learning rate.')
        self.model.load_state_dict(best_state)
        return self.history

    def evaluate(self, loader):
        self.model.eval()
        truth, predicted = [], []
        with torch.no_grad():
            for images, targets in loader:
                predicted.extend(self.model(images.to(self.device)).argmax(1).cpu().tolist())
                truth.extend(targets.tolist())
        if not truth:
            raise ValueError('Cannot evaluate an empty loader.')
        labels = list(range(len(self.class_names)))
        return {'accuracy': accuracy_score(truth, predicted),
                'balanced_accuracy': balanced_accuracy_score(truth, predicted),
                'macro_f1': f1_score(truth, predicted, labels=labels, average='macro', zero_division=0),
                'truth': truth, 'predicted': predicted,
                'classification_report': classification_report(truth, predicted, labels=labels,
                    target_names=self.class_names, output_dict=True, zero_division=0),
                'confusion_matrix': confusion_matrix(truth, predicted, labels=labels).tolist()}


## 5. Checkpoint inference

The checkpoint stores both preprocessing configuration and class order. Softmax scores rank known teams; they do not measure calibrated certainty or reject non-F1 photos.

In [ ]:
from PIL import Image, ImageOps
import torch



class TeamPredictor:
    """Classify one photo using saved class order and evaluation preprocessing."""

    def __init__(self, checkpoint, device='cpu'):
        self.device = torch.device(device)
        # weights_only avoids executing arbitrary pickled classes when restoring tensors.
        saved = torch.load(checkpoint, map_location=self.device, weights_only=True)
        self.class_names = saved['class_names']
        self.model = F1TeamCNN(len(self.class_names), saved['config']['dropout']).to(self.device)
        self.model.load_state_dict(saved['state_dict'])
        self.model.eval()  # Disable dropout for repeatable predictions.
        self.transform = F1ImageDataset([], saved['config']['image_size']).transform

    def predict(self, image_path, top_k=3):
        if top_k < 1:
            raise ValueError('top_k must be positive.')
        with Image.open(image_path) as source:
            image = ImageOps.exif_transpose(source).convert('RGB')
        with torch.no_grad():
            logits = self.model(self.transform(image).unsqueeze(0).to(self.device))
            scores, indices = logits.softmax(1)[0].topk(min(top_k, len(self.class_names)))
        # Softmax scores are model scores, not calibrated certainty.
        return [{'team': self.class_names[i], 'score': float(score)}
                for score, i in zip(scores.cpu(), indices.cpu().tolist())]


## 6. Figures and tables

Figures come from actual samples and measured results. Exploratory visualisation uses training images. Test error analysis is produced only after model selection.

In [ ]:
"""Save figures and tables from measured data; no invented results."""

from collections import Counter
import csv
from pathlib import Path
import os

# Keep font caches local so restricted Windows/PyCharm environments remain writable.
os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.cache' / 'matplotlib'))

import matplotlib
matplotlib.use('Agg')  # Saving plots works in PyCharm, notebooks and headless test runs.
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
from sklearn.metrics import ConfusionMatrixDisplay


def save_figure(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.close(fig)


def dataset_figures(audit, splits, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    names = audit.class_names
    fig, ax = plt.subplots(figsize=(11, 5))
    positions = np.arange(len(names))
    bottom = np.zeros(len(names))
    for split, records in splits.items():
        counts = Counter(r.team for r in records)
        values = np.array([counts[n] for n in names])
        ax.bar(positions, values, bottom=bottom, label=split)
        bottom += values
    ax.set_xticks(positions, names, rotation=35, ha='right')
    ax.set_ylabel('Images retained after audit')
    ax.legend()
    save_figure(fig, output / 'class_distribution.png')
    fig, axes = plt.subplots(len(names), 3, figsize=(10, 2.4 * len(names)), squeeze=False)
    for row, name in enumerate(names):
        # Use training examples in exploratory figures, keeping the test set unseen.
        records = [r for r in splits['train'] if r.team == name][:3]
        for ax, record in zip(axes[row], records):
            with Image.open(record.path) as source:
                ax.imshow(ImageOps.exif_transpose(source).convert('RGB'))
            ax.set_title(name, fontsize=9)
            ax.axis('off')
    save_figure(fig, output / 'training_examples.png')
    dimensions, means = [], []
    for record in splits['train']:
        with Image.open(record.path) as source:
            img = ImageOps.exif_transpose(source).convert('RGB')
            dimensions.append(img.size)
            means.append(np.asarray(img.resize((16, 16))).mean(axis=(0, 1)) / 255)
    dims = np.asarray(dimensions)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].scatter(dims[:, 0], dims[:, 1], s=8, alpha=.4)
    axes[0].set(xlabel='Original width', ylabel='Original height', title='Training image sizes')
    axes[1].hist(dims[:, 0] / dims[:, 1], bins=30)
    axes[1].set(xlabel='Width / height', ylabel='Count', title='Training aspect ratios')
    save_figure(fig, output / 'image_dimensions.png')
    fig, ax = plt.subplots(figsize=(5, 4))
    corr = np.corrcoef(np.asarray(means), rowvar=False)
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
    ax.set_xticks(range(3), ['Red', 'Green', 'Blue'])
    ax.set_yticks(range(3), ['Red', 'Green', 'Blue'])
    ax.set_title('Correlation of mean RGB values (training only)')
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center')
    fig.colorbar(im, ax=ax)
    save_figure(fig, output / 'rgb_correlation.png')


def architecture_figure(model, image_size, num_classes, output):
    stages = [f'RGB image\n3 x {image_size} x {image_size}',
              f'Conv 3x3 + ReLU (twice)\n32 channels; MaxPool 2x2\n32 x {image_size//2} x {image_size//2}',
              f'Conv 3x3 + ReLU (twice)\n64 channels; MaxPool 2x2\n64 x {image_size//4} x {image_size//4}',
              f'Conv 3x3 + ReLU (twice)\n128 channels; MaxPool 2x2\n128 x {image_size//8} x {image_size//8}',
              'Global average pool + flatten\n128 features',
              f'Dropout + Linear\n{num_classes} logits (one per team)']
    fig, ax = plt.subplots(figsize=(8, 9))
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.axis('off')
    for i, label in enumerate(stages):
        y = .92 - i * .16
        ax.text(.5, y, label, ha='center', va='center', fontsize=11,
                bbox=dict(boxstyle='round,pad=0.6', fc='#edf3fb', ec='#345983'))
        if i < len(stages)-1:
            ax.annotate('', xy=(.5, y-.10), xytext=(.5, y-.055), arrowprops={'arrowstyle': '->'})
    count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    ax.set_title(f'Explicit VGG-inspired CNN | {count:,} trainable parameters', pad=20)
    save_figure(fig, Path(output) / 'architecture.png')


def learning_curves(history, output):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for prefix, label in [('train', 'Training'), ('validation', 'Validation')]:
        for ax, metric in zip(axes, ['loss', 'accuracy']):
            ax.plot([r['epoch'] for r in history], [r[f'{prefix}_{metric}'] for r in history], label=label)
            ax.set(xlabel='Epoch', ylabel=metric.title())
            ax.legend()
    save_figure(fig, Path(output) / 'learning_curves.png')


def evaluation_figures(metrics, names, records, output, prefix='test'):
    output = Path(output)
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(np.array(metrics['confusion_matrix']), display_labels=names).plot(
        ax=ax, xticks_rotation=45, colorbar=False, cmap='Blues')
    save_figure(fig, output / f'{prefix}_confusion_matrix.png')
    with (output / f'{prefix}_predictions.csv').open('w', newline='', encoding='utf-8') as stream:
        writer = csv.writer(stream)
        writer.writerow(['path', 'actual_team', 'predicted_team', 'correct'])
        for record, actual, predicted in zip(records, metrics['truth'], metrics['predicted']):
            writer.writerow([str(record.path), names[actual], names[predicted], actual == predicted])
    errors = [(r, p) for r, a, p in zip(records, metrics['truth'], metrics['predicted']) if a != p][:12]
    if errors:
        fig, axes = plt.subplots((len(errors)+2)//3, 3, figsize=(11, 3*((len(errors)+2)//3)), squeeze=False)
        for ax in axes.flat:
            ax.axis('off')
        for ax, (record, predicted) in zip(axes.flat, errors):
            with Image.open(record.path) as source:
                ax.imshow(ImageOps.exif_transpose(source).convert('RGB'))
            ax.set_title(f'True: {record.team}\nPredicted: {names[predicted]}', fontsize=9)
        save_figure(fig, output / f'{prefix}_errors.png')


## 7. Controlled experiments and cross-validation

Use a fixed stratified 70/15/15 split. Compare baseline and augmentation using identical seeds and maximum training budgets. Select by validation macro F1, then accuracy. Run three folds on development data with fresh weights, then evaluate the selected checkpoint once on the held-out test set. Development CV follows selection and is not nested CV.

In [ ]:
"""Reproducible baseline, augmentation, development CV and held-out evaluation."""

from collections import Counter
from dataclasses import replace
from datetime import datetime, timezone
import importlib.metadata
import json
from pathlib import Path
import platform

import numpy as np
import torch



def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2), encoding='utf-8')


def train_experiment(records, train, validation, config, names, output, device=None):
    # Reset before construction so both experiments start from the same initial weights.
    seed_everything(config.seed)
    data = F1DataModule(records, config)
    trainer = Trainer(F1TeamCNN(len(names), config.dropout), config, names, device=device)
    history = trainer.fit(data.loader(train, True), data.loader(validation), output)
    metrics = trainer.evaluate(data.loader(validation))
    save_json(Path(output) / 'validation_metrics.json', metrics)
    save_json(Path(output) / 'config.json', config.to_dict())
    learning_curves(history, output)
    return trainer, metrics


def run_suite(data_root, output, config=None, folds=3, device=None):
    config = config or TrainingConfig()
    if folds < 2:
        raise ValueError('At least two development folds are required; the default is three.')
    output = Path(output)
    if output.exists():
        raise FileExistsError(f'Choose a new output directory to preserve the previous run: {output}')
    audit = DatasetAudit(data_root).scan()
    data = F1DataModule(audit.records, config)
    splits = data.split()
    # Validate all folds before starting any potentially lengthy experiment.
    development = splits['train'] + splits['validation']
    development_folds = list(data.folds(development, folds))
    output.mkdir(parents=True)
    save_json(output / 'audit.json', audit.summary())
    save_json(output / 'split_manifest.json', {
        split: [{'path': r.path.resolve().as_posix(), 'team': r.team, 'label': r.label, 'digest': r.digest}
                for r in records] for split, records in splits.items()})
    versions = {name: importlib.metadata.version(name) for name in
                ['torch', 'torchvision', 'numpy', 'Pillow', 'scikit-learn', 'matplotlib']}
    save_json(output / 'environment.json', {'python': platform.python_version(), 'packages': versions,
        'device': device or ('cuda' if torch.cuda.is_available() else 'cpu'),
        'utc_started': datetime.now(timezone.utc).isoformat(), 'config': config.to_dict()})
    print(f'Audit retained {len(audit.records)} images in {len(audit.class_names)} teams.', flush=True)
    print({name: len(records) for name, records in splits.items()}, flush=True)
    figures = output / 'figures'
    dataset_figures(audit, splits, figures)
    seed_everything(config.seed)
    architecture_figure(F1TeamCNN(len(audit.class_names), config.dropout), config.image_size,
                        len(audit.class_names), figures)
    candidates = {}
    for name, augment in [('baseline', False), ('augmented', True)]:
        print(f'Experiment: {name}', flush=True)
        experiment_config = replace(config, augment=augment)
        trainer, metrics = train_experiment(audit.records, splits['train'], splits['validation'],
                                            experiment_config, audit.class_names, output / name, device)
        candidates[name] = {'trainer': trainer, 'metrics': metrics, 'config': experiment_config}
    # The test set has not been used: selection uses macro F1 on validation only.
    selected = max(candidates, key=lambda name: (candidates[name]['metrics']['macro_f1'],
                                                candidates[name]['metrics']['accuracy']))
    selected_config = candidates[selected]['config']
    save_json(output / 'selection.json', {'selected': selected, 'criterion': 'validation macro F1, then accuracy',
        'experiments': {n: {k: v['metrics'][k] for k in ['accuracy', 'balanced_accuracy', 'macro_f1']}
                        for n, v in candidates.items()}})
    scores = []
    for fold, (train, validation) in enumerate(development_folds, 1):
        print(f'Development cross-validation: {fold}/{folds}, configuration {selected}', flush=True)
        _, metrics = train_experiment(audit.records, train, validation, selected_config,
                                      audit.class_names, output / f'cv_fold_{fold}', device)
        scores.append({k: metrics[k] for k in ['accuracy', 'balanced_accuracy', 'macro_f1']})
        save_json(output / f'cv_fold_{fold}' / 'manifest.json', {
            'train': [r.digest for r in train], 'validation': [r.digest for r in validation]})
    cv = {'configuration': selected, 'folds': scores,
          'mean': {k: float(np.mean([r[k] for r in scores])) for k in scores[0]},
          'std': {k: float(np.std([r[k] for r in scores], ddof=1)) for k in scores[0]},
          'note': 'Development CV after configuration selection; not nested CV or an unbiased final estimate.'}
    save_json(output / 'cross_validation.json', cv)
    # Evaluate the already selected checkpoint once; do not tune against this result.
    best = candidates[selected]['trainer']
    test = best.evaluate(F1DataModule(audit.records, selected_config).loader(splits['test']))
    save_json(output / 'test_metrics.json', test)
    evaluation_figures(test, audit.class_names, splits['test'], figures)
    majority = Counter(r.label for r in splits['train']).most_common(1)[0][0]
    majority_accuracy = sum(r.label == majority for r in splits['test']) / len(splits['test'])
    summary = {'selected_experiment': selected, 'checkpoint': f'{selected}/best.pt',
               'class_names': audit.class_names, 'data_root': str(Path(data_root).resolve()),
               'images': len(audit.records), 'split_sizes': {k: len(v) for k, v in splits.items()},
               'test_accuracy': test['accuracy'], 'test_macro_f1': test['macro_f1'],
               'test_balanced_accuracy': test['balanced_accuracy'],
               'majority_baseline_accuracy': majority_accuracy, 'cv': cv,
               'best_epoch': best.best_epoch, 'completed_utc': datetime.now(timezone.utc).isoformat()}
    save_json(output / 'summary.json', summary)
    # A latest pointer is written only after every experiment and final evaluation succeeds.
    save_json(output.parent / 'latest.json', {'run': str(output.resolve()),
                                            'checkpoint': str((output / summary['checkpoint']).resolve())})
    print(json.dumps(summary, indent=2), flush=True)
    return summary


## 8. Run or load the experiments

The cleaned dataset must be a directory of team folders. The reported friend-cleaning counts do not certify that the current DATA_ROOT is that version. Every completed run records its actual source path, audit and split hashes. New training uses a unique run directory.

In [ ]:
config = TrainingConfig(epochs=20, image_size=64, patience=6)
if RUN_TRAINING:
    from datetime import datetime
    RUN_DIR = PROJECT_ROOT / 'runs' / datetime.now().strftime('notebook_%Y%m%d_%H%M%S_%f')
    summary = run_suite(DATA_ROOT, RUN_DIR, config, folds=3)
elif RUN_DIR is None and (PROJECT_ROOT / 'runs' / 'latest.json').exists():
    RUN_DIR = Path(json.loads((PROJECT_ROOT / 'runs' / 'latest.json').read_text())['run'])
if RUN_DIR is not None:
    RUN_DIR = Path(RUN_DIR)
    summary = json.loads((RUN_DIR / 'summary.json').read_text())
    print(json.dumps(summary, indent=2))
    print('These results used:', summary['data_root'])
else:
    print('No completed run found. Set DATA_ROOT and RUN_TRAINING=True, or run train.py first.')


## 9. Dataset and architecture evidence

Describe class imbalance, resolution, RGB correlations and possible background shortcuts. Check the actual input size and configuration of this run.

In [ ]:
if RUN_DIR is not None:
    audit_result = json.loads((RUN_DIR / 'audit.json').read_text())
    print('Class counts:', audit_result['class_counts'])
    print('Invalid:', len(audit_result['invalid_files']), 'duplicate copies:', len(audit_result['removed_duplicates']))
    for name in ['class_distribution', 'training_examples', 'image_dimensions', 'rgb_correlation', 'architecture']:
        display(DisplayImage(filename=str(RUN_DIR / 'figures' / f'{name}.png')))


## 10. Experimental results

Compare both learning curves and validation metrics. Discuss the realised epoch budgets, overfitting or underfitting, and whether augmentation helped. Use the reported cross-validation standard deviation as descriptive variability, not a confidence interval.

In [ ]:
if RUN_DIR is not None:
    print((RUN_DIR / 'selection.json').read_text())
    print((RUN_DIR / 'cross_validation.json').read_text())
    for name in ['baseline', 'augmented']:
        print(name, (RUN_DIR / name / 'config.json').read_text())
        display(DisplayImage(filename=str(RUN_DIR / name / 'learning_curves.png')))


## 11. Held-out evaluation

These are the test results of the checkpoint chosen using validation. Do not select new hyperparameters after inspecting them. Report per-team precision, recall and F1 alongside overall and balanced accuracy; compare against the majority-class baseline.

In [ ]:
if RUN_DIR is not None:
    metrics = json.loads((RUN_DIR / 'test_metrics.json').read_text())
    print(json.dumps({k:v for k,v in metrics.items() if k not in ['truth','predicted']}, indent=2))
    display(DisplayImage(filename=str(RUN_DIR / 'figures' / 'test_confusion_matrix.png')))
    errors = RUN_DIR / 'figures' / 'test_errors.png'
    if errors.exists():
        display(DisplayImage(filename=str(errors)))


## 12. Predict a photo

Run `app.py` for the image picker, or set PHOTO_PATH above. Historical dataset labels are preserved; the classifier does not automatically support current teams or unseen liveries.

In [ ]:
if PHOTO_PATH is not None:
    if RUN_DIR is None:
        raise ValueError('Train a model or select a completed RUN_DIR first.')
    predictor = TeamPredictor(RUN_DIR / summary['checkpoint'])
    print(predictor.predict(PHOTO_PATH))
    display(DisplayImage(filename=str(PHOTO_PATH)))


## 13. Submission checks

Use `docs/REPORT_CHECKLIST.md` and `docs/REFERENCES.md`. Write the PDF sections in your own voice, including difficulty and individual contributions. Complete `AI_USAGE.md` with all prompts. Add your own critical-code explanations and rehearse a walkthrough. Confirm the brief's Assign2 filename discrepancy, insert actual IDs, then restart and Run All in the submission environment.